# Exercise 2: Master vs Single - Performance Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## 1. Execution Time Comparison

In [ ]:
# Example data - replace with your measurements
# For N=1000 matrix
data = {
    'threads': [1, 2, 4, 8],
    'serial_time': [0.008, 0.008, 0.008, 0.008],
    'parallel_time': [0.009, 0.005, 0.003, 0.003],  # Replace with actual
    'init_time': [0.003, 0.003, 0.003, 0.003],  # Serial portion
    'print_time': [0.002, 0.002, 0.002, 0.002],  # Serial portion
    'sum_time': [0.003, 0.0015, 0.0008, 0.0004]  # Parallel portion
}

df = pd.DataFrame(data)
df['speedup'] = df['serial_time'] / df['parallel_time']
df['efficiency'] = (df['speedup'] / df['threads']) * 100
df

## 2. Amdahl's Law Visualization

In [ ]:
# Calculate serial fraction
T_serial_portion = data['init_time'][0] + data['print_time'][0]
T_total = data['serial_time'][0]
f_serial = T_serial_portion / T_total  # Serial fraction
f_parallel = 1 - f_serial

print(f"Serial fraction: {f_serial:.2%}")
print(f"Parallel fraction: {f_parallel:.2%}")

# Amdahl's Law: Speedup = 1 / (f_serial + f_parallel/p)
threads_range = np.arange(1, 17)
amdahl_speedup = 1 / (f_serial + f_parallel / threads_range)

fig, ax = plt.subplots(figsize=(10, 6))

# Plot theoretical Amdahl's Law
ax.plot(threads_range, amdahl_speedup, '--', label=f'Amdahl\'s Law (f_s={f_serial:.2%})', 
        linewidth=2, color='red', alpha=0.7)

# Plot actual measurements
ax.plot(df['threads'], df['speedup'], 'o-', label='Actual Speedup', 
        linewidth=2, markersize=8, color='blue')

# Plot ideal linear speedup
ax.plot(threads_range, threads_range, ':', label='Ideal (Linear)', 
        linewidth=2, color='green', alpha=0.5)

# Add theoretical limit
max_speedup = 1 / f_serial
ax.axhline(y=max_speedup, linestyle='-.', label=f'Theoretical Max ({max_speedup:.2f}x)', 
           color='orange', linewidth=2)

ax.set_xlabel('Number of Threads', fontsize=12)
ax.set_ylabel('Speedup', fontsize=12)
ax.set_title('Speedup vs Amdahl\'s Law Prediction', fontsize=14, weight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, 16)

plt.tight_layout()
plt.savefig('ex2_amdahl.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTheoretical maximum speedup: {max_speedup:.2f}x")

## 3. Time Breakdown (Stacked Bar Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = df['threads']
width = 0.6

# Stacked bars
p1 = ax.bar(x, df['init_time'], width, label='Init (Master)', color='#e74c3c')
p2 = ax.bar(x, df['print_time'], width, bottom=df['init_time'], 
            label='Print (Single)', color='#f39c12')
p3 = ax.bar(x, df['sum_time'], width, 
            bottom=df['init_time'] + df['print_time'],
            label='Sum (Parallel)', color='#2ecc71')

ax.set_xlabel('Number of Threads', fontsize=12)
ax.set_ylabel('Time (s)', fontsize=12)
ax.set_title('Execution Time Breakdown', fontsize=14, weight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Add total time labels
for i, (thread, total) in enumerate(zip(df['threads'], df['parallel_time'])):
    ax.text(thread, total + 0.0002, f'{total:.4f}s', 
            ha='center', va='bottom', fontsize=9, weight='bold')

plt.tight_layout()
plt.savefig('ex2_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Efficiency Analysis

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Speedup
ax1.plot(df['threads'], df['speedup'], 'o-', linewidth=2, markersize=8, color='#3498db')
ax1.plot(df['threads'], df['threads'], '--', linewidth=2, color='gray', alpha=0.5, label='Linear')
ax1.set_xlabel('Number of Threads', fontsize=12)
ax1.set_ylabel('Speedup', fontsize=12)
ax1.set_title('Speedup', fontsize=14, weight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Efficiency
ax2.plot(df['threads'], df['efficiency'], 'o-', linewidth=2, markersize=8, color='#9b59b6')
ax2.axhline(y=100, linestyle='--', linewidth=2, color='gray', alpha=0.5, label='Ideal (100%)')
ax2.set_xlabel('Number of Threads', fontsize=12)
ax2.set_ylabel('Efficiency (%)', fontsize=12)
ax2.set_title('Parallel Efficiency', fontsize=14, weight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 110)

plt.tight_layout()
plt.savefig('ex2_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Master vs Single Comparison

In [ ]:
# Conceptual comparison
fig, ax = plt.subplots(figsize=(10, 6))

directives = ['master', 'single']
properties = ['Thread\nSelection', 'Implicit\nBarrier', 'Use for\nI/O', 'Flexibility']

master_scores = [1, 1, 5, 3]   # Thread 0 only, No barrier, Good for I/O, Less flexible
single_scores = [5, 5, 4, 5]   # Any thread, Has barrier, OK for I/O, More flexible

x = np.arange(len(properties))
width = 0.35

bars1 = ax.bar(x - width/2, master_scores, width, label='master', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, single_scores, width, label='single', color='#3498db', alpha=0.8)

ax.set_xlabel('Property', fontsize=12)
ax.set_ylabel('Score (1=Restrictive, 5=Flexible)', fontsize=12)
ax.set_title('Master vs Single Directive Comparison', fontsize=14, weight='bold')
ax.set_xticks(x)
ax.set_xticklabels(properties)
ax.legend()
ax.set_ylim(0, 6)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('ex2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Key Takeaways

### Observations:
1. **Serial bottleneck**: Init and print limit speedup
2. **Amdahl's Law**: Matches theoretical prediction
3. **Diminishing returns**: Beyond 4 threads, little improvement

### When to use:
- **`master`**: Master-specific I/O, no barrier needed
- **`single`**: One-time work, barrier needed

### Critical reminder:
**Always add explicit barrier after `master` if other threads need the result!**